# Fine-tune PhoBERT ViQuAD trên Kaggle

Notebook này fine-tune checkpoint [`heidiie/phobert_finetuned_viquad`](https://huggingface.co/heidiie/phobert_finetuned_viquad) bằng pipeline Reader đã audit của VIQA Nexus.

Trước khi chạy: mở **Notebook options**, chọn **GPU P100/T4** và bật **Internet**. Sau đó chọn **Run all**. Notebook dùng dữ liệu sạch trong nhánh `23021677`, chọn checkpoint bằng decoded `answerable_f1`, calibrate ngưỡng no-answer trên toàn bộ validation và xuất model `.zip`.

> Không dùng tập test để chọn model. Tập validation 3.814 câu là tập có nhãn dùng cho EM/F1 và threshold calibration.

In [ ]:
# 1. Cấu hình thí nghiệm
REPO_URL = "https://github.com/duong158/QA_system.git"
BRANCH = "23021677"
BASE_MODEL = "heidiie/phobert_finetuned_viquad"
RUN_NAME = "phobert_viquad_kaggle_ft"

EPOCHS = 2
LEARNING_RATE = 1e-5
TRAIN_BATCH_SIZE = 8
EVAL_BATCH_SIZE = 16
GRADIENT_ACCUMULATION_STEPS = 2
MAX_SEQ_LEN = 256
DOC_STRIDE = 80
SEED = 42

# True: chạy thử 300 mẫu; False: train/evaluate đầy đủ để lấy model cuối.
SMOKE_TEST = False
SMOKE_SUBSET_SIZE = 300

# Baseline 300 mẫu giúp phát hiện lỗi tải model trước khi tốn thời gian train.
RUN_BASELINE_SMOKE_EVAL = True
RUN_SPAN_VALIDATION = True

print({
    "base_model": BASE_MODEL,
    "epochs": EPOCHS,
    "learning_rate": LEARNING_RATE,
    "smoke_test": SMOKE_TEST,
})

In [ ]:
# 2. Clone đúng nhánh project
import os
import subprocess
from pathlib import Path

PROJECT_DIR = Path("/kaggle/working/QA_system")

if PROJECT_DIR.exists() and not (PROJECT_DIR / ".git").exists():
    raise RuntimeError(f"{PROJECT_DIR} đã tồn tại nhưng không phải Git repository. Hãy đổi PROJECT_DIR.")

if not PROJECT_DIR.exists():
    subprocess.run(
        ["git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, str(PROJECT_DIR)],
        check=True,
    )
else:
    subprocess.run(["git", "-C", str(PROJECT_DIR), "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", str(PROJECT_DIR), "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", str(PROJECT_DIR), "pull", "--ff-only", "origin", BRANCH], check=True)

os.chdir(PROJECT_DIR)
commit = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip()
print(f"Project: {PROJECT_DIR}")
print(f"Branch:  {BRANCH}")
print(f"Commit:  {commit}")

In [ ]:
# 3. Cài dependency Reader (giữ PyTorch GPU có sẵn của Kaggle)
import sys

packages = [
    "transformers>=4.46,<5",
    "datasets>=2.18,<4",
    "accelerate>=0.26,<2",
    "huggingface_hub>=0.23",
    "pyvi",
    "scikit-learn",
    "pandas",
    "pyarrow",
]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *packages], check=True)
print("Đã cài xong dependency.")

In [ ]:
# 4. Bắt buộc xác nhận đang dùng GPU
subprocess.run(["nvidia-smi"], check=False)

import torch
import transformers
import datasets

assert torch.cuda.is_available(), (
    "Không thấy CUDA. Trong Kaggle hãy chọn Settings/Notebook options -> Accelerator -> GPU."
)
gpu_name = torch.cuda.get_device_name(0)
gpu_memory_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f"PyTorch: {torch.__version__}")
print(f"Transformers: {transformers.__version__}; Datasets: {datasets.__version__}")
print(f"GPU: {gpu_name} ({gpu_memory_gb:.1f} GB)")

if gpu_memory_gb < 12:
    print("CẢNH BÁO: VRAM thấp. Nên đặt TRAIN_BATCH_SIZE=4, EVAL_BATCH_SIZE=8, GRADIENT_ACCUMULATION_STEPS=4 rồi chạy lại từ đầu.")

In [ ]:
# 5. Kiểm tra dữ liệu sạch trước khi train
import pandas as pd

train_file = PROJECT_DIR / "data/processed/viquad_train_clean.parquet"
val_file = PROJECT_DIR / "data/processed/viquad_val_clean.parquet"
assert train_file.is_file(), f"Thiếu {train_file}"
assert val_file.is_file(), f"Thiếu {val_file}"

train_df = pd.read_parquet(train_file)
val_df = pd.read_parquet(val_file)
required_columns = {"id", "title", "context", "question", "answer_text", "answer_start"}
assert required_columns.issubset(train_df.columns), required_columns - set(train_df.columns)
assert required_columns.issubset(val_df.columns), required_columns - set(val_df.columns)
assert len(train_df) == 28454, f"Train phải có 28.454 mẫu, hiện có {len(train_df)}"
assert len(val_df) == 3814, f"Validation phải có 3.814 mẫu, hiện có {len(val_df)}"

summary = pd.DataFrame([
    {"split": "train", "rows": len(train_df), "answerable": int((train_df.answer_start >= 0).sum())},
    {"split": "validation", "rows": len(val_df), "answerable": int((val_df.answer_start >= 0).sum())},
])
display(summary)
display(train_df[["question", "answer_text", "answer_start"]].head(3))

In [ ]:
# 6. Tải đầy đủ checkpoint + tokenizer từ Hugging Face
from huggingface_hub import snapshot_download
from transformers import AutoConfig, AutoModelForQuestionAnswering
from reader.data_utils import get_tokenizer

cached_model_dir = Path(snapshot_download(repo_id=BASE_MODEL))
required_model_files = ["config.json", "model.safetensors", "vocab.txt", "bpe.codes"]
missing_model_files = [name for name in required_model_files if not (cached_model_dir / name).is_file()]
assert not missing_model_files, f"Checkpoint thiếu file: {missing_model_files}"

config = AutoConfig.from_pretrained(BASE_MODEL)
tokenizer = get_tokenizer(BASE_MODEL)
probe_model = AutoModelForQuestionAnswering.from_pretrained(BASE_MODEL)
assert probe_model.config.vocab_size == len(tokenizer), (
    f"Vocab model/tokenizer không khớp: {probe_model.config.vocab_size} != {len(tokenizer)}"
)
print(f"Checkpoint cache: {cached_model_dir}")
print(f"Architecture: {config.architectures}; vocab_size={config.vocab_size}")
del probe_model
torch.cuda.empty_cache()

In [ ]:
# 7. Smoke evaluation checkpoint gốc (300 mẫu, để phát hiện lỗi pipeline)
baseline_dir = PROJECT_DIR / "results/reader/baseline_public_smoke"
if RUN_BASELINE_SMOKE_EVAL:
    baseline_cmd = [
        sys.executable, "-m", "reader.evaluate",
        "--model_path", BASE_MODEL,
        "--subset_size", str(SMOKE_SUBSET_SIZE),
        "--batch_size", str(EVAL_BATCH_SIZE),
        "--max_seq_len", str(MAX_SEQ_LEN),
        "--doc_stride", str(DOC_STRIDE),
        "--output_dir", str(baseline_dir),
    ]
    print("Running:", " ".join(baseline_cmd))
    subprocess.run(baseline_cmd, cwd=PROJECT_DIR, check=True)
else:
    print("Bỏ qua baseline smoke evaluation.")

In [ ]:
# 8. Xác minh answer_start đi qua PyVi + PhoBERT BPE đúng vị trí
# Khi SMOKE_TEST=False, kiểm tra toàn bộ train + validation trước khi train.
if RUN_SPAN_VALIDATION:
    span_cmd = [
        sys.executable, "-m", "reader.validate_spans",
        "--model", BASE_MODEL,
        "--splits", "train", "validation",
        "--max_length", str(MAX_SEQ_LEN),
        "--stride", str(DOC_STRIDE),
        "--batch_size", "128",
        "--output", "results/kaggle_span_integrity_report.json",
        "--errors", "results/kaggle_span_integrity_errors.csv",
    ]
    if SMOKE_TEST:
        span_cmd += ["--subset_size", str(SMOKE_SUBSET_SIZE)]
    print("Running:", " ".join(span_cmd))
    subprocess.run(span_cmd, cwd=PROJECT_DIR, check=True)
else:
    print("Bỏ qua span validation.")

In [ ]:
# 9. Fine-tune checkpoint trên dữ liệu sạch
train_cmd = [
    sys.executable, "-m", "reader.train",
    "--model_name", BASE_MODEL,
    "--run_name", RUN_NAME,
    "--output_dir", "models/reader/kaggle",
    "--results_root", "results/reader/kaggle",
    "--epochs", str(EPOCHS),
    "--lr", str(LEARNING_RATE),
    "--batch_size", str(TRAIN_BATCH_SIZE),
    "--eval_batch_size", str(EVAL_BATCH_SIZE),
    "--gradient_accumulation_steps", str(GRADIENT_ACCUMULATION_STEPS),
    "--max_seq_len", str(MAX_SEQ_LEN),
    "--doc_stride", str(DOC_STRIDE),
    "--seed", str(SEED),
]
if SMOKE_TEST:
    train_cmd += ["--subset_size", str(SMOKE_SUBSET_SIZE)]

print("Running:", " ".join(train_cmd))
subprocess.run(train_cmd, cwd=PROJECT_DIR, check=True)

In [ ]:
# 10. Đọc kết quả validation và threshold đã calibrate
import json

MODEL_DIR = PROJECT_DIR / "models/reader/kaggle" / RUN_NAME
RESULT_DIR = PROJECT_DIR / "results/reader/kaggle" / RUN_NAME
metrics_file = RESULT_DIR / "validation_metrics.json"
threshold_file = RESULT_DIR / "best_threshold.json"
assert (MODEL_DIR / "model.safetensors").is_file(), f"Không tìm thấy model tại {MODEL_DIR}"
assert metrics_file.is_file(), f"Không tìm thấy metrics tại {metrics_file}"
assert threshold_file.is_file(), f"Không tìm thấy threshold tại {threshold_file}"

metrics = json.loads(metrics_file.read_text(encoding="utf-8"))
threshold = json.loads(threshold_file.read_text(encoding="utf-8"))
metric_summary = pd.DataFrame([{
    "overall_em": metrics["overall"]["em"],
    "overall_f1": metrics["overall"]["f1"],
    "answerable_em": metrics["answerable"]["em"],
    "answerable_f1": metrics["answerable"]["f1"],
    "unanswerable_accuracy": metrics["unanswerable"]["accuracy"],
    "best_threshold": threshold["threshold"],
}])
display(metric_summary)
print(f"Model: {MODEL_DIR}")
print(f"Artifacts: {RESULT_DIR}")

In [ ]:
# 11. Thử suy luận bằng đúng production Reader
from reader.predict import ReaderPredictor

predictor = ReaderPredictor(str(MODEL_DIR))
question = "Công trình nào nằm trên đỉnh Montmartre?"
context = (
    "Montmartre là một khu phố nổi tiếng ở Paris. "
    "Vương cung thánh đường Sacré-Cœur nằm trên đỉnh đồi Montmartre và nhìn ra thành phố."
)
prediction = predictor.predict(
    question,
    context,
    max_seq_len=MAX_SEQ_LEN,
    doc_stride=DOC_STRIDE,
)
display(prediction)

In [ ]:
# 12. Nén model và kết quả để tải từ Kaggle Output
import shutil

model_zip_base = Path("/kaggle/working") / f"{RUN_NAME}_model"
results_zip_base = Path("/kaggle/working") / f"{RUN_NAME}_results"
model_zip = shutil.make_archive(str(model_zip_base), "zip", root_dir=MODEL_DIR)
results_zip = shutil.make_archive(str(results_zip_base), "zip", root_dir=RESULT_DIR)

print(f"Model ZIP:   {model_zip}")
print(f"Results ZIP: {results_zip}")
print("Mở panel Files bên phải để tải hai file ZIP về máy.")

## Cách chọn model cuối

- Chỉ dùng kết quả khi `SMOKE_TEST=False`; smoke test không phải benchmark cuối.
- So sánh `answerable_f1`, `overall_f1` và `unanswerable_accuracy` với checkpoint production hiện tại.
- Không copy model vào production nếu F1 giảm hoặc tỷ lệ câu answerable bị trả rỗng tăng bất thường.
- File `best_reader_config.json` bên trong model chứa ngưỡng no-answer đã calibrate và phải được triển khai cùng `model.safetensors`, `config.json`, `vocab.txt`, `bpe.codes`.